# Fase 4: ventanas y características

Convierte la tabla integrada (una fila cada 10 s) en una tabla de **ventanas**: una
fila por ventana de tiempo, con unas pocas **características** que la resumen y el
**nivel de ocupación** de esa ventana. Esa tabla es la que aprende el modelo.

Diapositivas: clase 7, 46 (Attribute Creation / Feature Generation) y 47 (Data Transformation: aggregation).

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

ROOT = Path.cwd()
while not (ROOT / "config.yaml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from occupancy.cleaning import clean
from occupancy.integration import merge_all

with open(ROOT / "config.yaml", encoding="utf-8") as f:
    config = yaml.safe_load(f)
with open(ROOT / "data" / "simulated" / "readings.json", encoding="utf-8") as f:
    data = json.load(f)["data"]

# Fases 1 y 2
tables, pir, labels = clean(data, config["cleaning"])
df = merge_all(tables, pir, labels)

window = f"{config['windows']['size_minutes']}min"
step = config["windows"]["step_seconds"] // config["cleaning"]["interval_seconds"]
sensors = list(tables)
pir_columns = ["pir-1", "pir-2", "pir-3"]
print("Tabla integrada:", df.shape, "| ventana:", window, "| paso:", step, "filas")

Tabla integrada: (1080, 13) | ventana: 3min | paso: 3 filas


## Paso 1: ventanas móviles y medias

In [2]:
# Tu código aquí
roll = df.rolling(window, min_periods=1)
features = pd.DataFrame(index=df.index)
for sensor in sensors:
    features[f"{sensor}_mean"] = roll[sensor].mean()

## Paso 2: características específicas de cada sensor

In [3]:
# Tu código aquí
features["co2Equivalent_change"] = df["co2Equivalent"] - df["co2Equivalent"].shift(freq=window)
features["mic_vol_max"] = roll["mic_vol"].max()
features["mic_vol_std"] = roll["mic_vol"].std()

pir_activity = df[pir_columns].mean(axis=1)
features["pir_mean"] = pir_activity.rolling(window).mean()
features["pir_max"] = df[pir_columns].max(axis=1).rolling(window).max()

## Paso 3: la etiqueta de cada ventana

In [4]:
# Tu código aquí
def most_frequent(values):
    values = pd.Series(values).dropna()
    if values.empty:
        return np.nan
    return values.mode().iloc[0]

features["label"] = df["busynessLabel"].rolling(window).apply(most_frequent, raw=True)

## Paso 4: quedarse con una ventana cada 30 s y solo las válidas

In [5]:
# Tu código aquí
coverage = df["temperature"].notna().rolling(window).mean()

windows = features.iloc[::step]
coverage = coverage.iloc[::step]

complete = windows.index >= df.index[0] + pd.Timedelta(window)
valid = complete & (coverage >= 0.8) & windows["label"].notna()

windows = windows[valid].copy()
windows["label"] = windows["label"].astype(int)

In [6]:
# Comprobación (no hace falta modificarla)
checks = {
    "345 ventanas válidas": len(windows) == 345,
    "sin la etiqueta como característica": "busynessLabel" not in windows.columns,
    "niveles 0 a 3": set(windows["label"].unique()) <= {0, 1, 2, 3},
}
for name, ok in checks.items():
    print(f"{name:36} {'OK' if ok else 'REVISAR'}")
print(windows["label"].value_counts().sort_index())

345 ventanas válidas                 OK
sin la etiqueta como característica  OK
niveles 0 a 3                        OK
label
0    201
1     22
2     45
3     77
Name: count, dtype: int64
